# Практикалық жұмыс №4
## Тақырыбы: Жақын көрші алгоритмі негізінде классификаторды жүзеге асыру (K Neighbors Classifier)
**Орындаған:** [Имя удалено по соображениям конфиденциальности]  
**Тобы:** 6B06106 – Computer Science  
**Вариант:** 14 (нұсқа №4: Электрондық хаттың белгілері және спам/спам емес класы)

In [ ]:
# 1. Қажетті кітапханаларды жүктеу
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

In [ ]:
# 2. Деректерді жүктеу және алғашқы қарап шығу
df = pd.read_csv('data.csv')
print("Деректердің өлшемі:", df.shape)
print("
--- Алғашқы 5 жазба ---")
print(df.head())
print("
--- Деректер құрылымы туралы мәлімет ---")
print(df.info())

In [ ]:
# 3. Статистикалық сипаттамалар және деректерді алдын ала талдау (EDA)
print("--- Сипаттамалық статистика ---")
print(df.describe())
print("
Бос мәндер саны (isnull().sum()):\n", df.isnull().sum())
print("
Қайталанатын жазбалар саны:", df.duplicated().sum())
print("
Кластар үлестірімі (result):\n", df['result'].value_counts())

In [ ]:
# 4. Деректерді визуализациялау
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Хат ұзындығы мен сілтемелер саны бойынша шашырау диаграммасы
scatter = axes[0].scatter(df['text_length'], df['num_links'], c=df['result'], 
                          cmap='coolwarm', alpha=0.75, edgecolors='k', s=45)
axes[0].set_title('Хат ұзындығы және сілтемелер саны')
axes[0].set_xlabel('Мәтін ұзындығы (символ)')
axes[0].set_ylabel('Сілтемелер саны')
cbar = plt.colorbar(scatter, ax=axes[0], ticks=[0, 1])
cbar.set_ticklabels(['Спам емес (0)', 'Спам (1)'])

# Спам сөздердің таралуы
sns.boxplot(x='result', y='spam_word_count', data=df, ax=axes[1], palette=['#6baed6', '#fb6a4a'])
axes[1].set_title('Спам сөздер саны (spam_word_count)')
axes[1].set_xlabel('Мақсатты класс (0 - Қалыпты, 1 - Спам)')
axes[1].set_ylabel('Спам сөздер саны')
axes[1].set_xticklabels(['Қалыпты (0)', 'Спам (1)'])

plt.tight_layout()
plt.show()

In [ ]:
# 5. Белгілер мен мақсатты айнымалыны анықтау, бөлу және масштабтау
features = [
    'text_length', 'num_links', 'num_exclamations', 'num_uppercase_words',
    'spam_word_count', 'has_currency', 'is_reply', 'sender_in_contacts',
    'has_attachment', 'hour_sent'
]
X = df[features]
y = df['result']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Оқыту жиыны:", X_train_scaled.shape)
print("Тестілеу жиыны:", X_test_scaled.shape)

In [ ]:
# 6. Әртүрлі k мәндерінің әсерін зерттеу
k_range = list(range(1, 21))
accuracies = []

for k in k_range:
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_train_scaled, y_train)
    accuracies.append(clf.score(X_test_scaled, y_test))

plt.figure(figsize=(9, 4.5))
plt.plot(k_range, accuracies, marker='o', color='#1f77b4', linewidth=2)
plt.title('k параметрінің сапа көрсеткішіне (Accuracy) әсері')
plt.xlabel('k мәні (n_neighbors)')
plt.ylabel('Дәлдік (Accuracy)')
plt.xticks(k_range)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# 7. KNeighborsClassifier моделін оқыту және сапасын бағалау (k=5)
knn = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)
y_prob = knn.predict_proba(X_test_scaled)

print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:   ", recall_score(y_test, y_pred))
print("F1-score: ", f1_score(y_test, y_pred))
print("\n--- Classification Report ---\n", classification_report(y_test, y_pred, target_names=['Қалыпты (0)', 'Спам (1)']))

In [ ]:
# 8. Confusion Matrix визуализациясы
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Қалыпты (0)', 'Спам (1)'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix - KNeighborsClassifier (k=5)')
plt.show()

In [ ]:
# 9. Жаңа объектілер үшін болжам жасау және ықтималдықтарды есептеу
new_emails = pd.DataFrame([
    # 1. Қалыпты хат үлгісі
    {'text_length': 340, 'num_links': 1, 'num_exclamations': 0, 'num_uppercase_words': 0,
     'spam_word_count': 0, 'has_currency': 0, 'is_reply': 1, 'sender_in_contacts': 1,
     'has_attachment': 1, 'hour_sent': 11},
    # 2. Спам хат үлгісі
    {'text_length': 120, 'num_links': 4, 'num_exclamations': 3, 'num_uppercase_words': 3,
     'spam_word_count': 3, 'has_currency': 1, 'is_reply': 0, 'sender_in_contacts': 0,
     'has_attachment': 0, 'hour_sent': 2}
], columns=features)

new_scaled = scaler.transform(new_emails)
predictions = knn.predict(new_scaled)
probabilities = knn.predict_proba(new_scaled)

for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    label = "СПАМ (1)" if pred == 1 else "ҚАЛЫПТЫ ХАТ (0)"
    print(f"Хат #{i+1} бойынша болжам: {label}")
    print(f"  Ықтималдықтар: P(Спам емес) = {prob[0]:.2f}, P(Спам) = {prob[1]:.2f}\n")